# Train the Tilt Sketch AI

Runs the full pipeline on a free Colab GPU (about 30 minutes on a T4 GPU):
1. download Quick, Draw! drawings for the game's 78 objects plus an "other" pool,
2. render them with the same framing as the browser and gyro-style augmentation,
3. train the classifier, calibrate it and export it for the browser,
4. evaluate it against the previous AI (DoodleNet) and write `docs/EVALUATION.md`.

**Before running:** Runtime → Change runtime type → GPU.

In [ ]:
%cd /content
![ -d tilt-sketch ] && git -C tilt-sketch pull || git clone https://github.com/Ser2104/tilt-sketch.git
%cd /content/tilt-sketch
!pip -q install pillow opencv-python-headless

In [ ]:
%cd /content/tilt-sketch
!python training/prepare_data.py --per-class 5000

In [ ]:
%cd /content/tilt-sketch
# Optional: add the gyroscope drawings players shared. Store the project's secret key as a Colab secret named SUPABASE_SECRET_KEY
# (key icon in the left bar). Never paste it in a cell or commit it.
import os
try:
    from google.colab import userdata
    os.environ['SUPABASE_SECRET_KEY'] = userdata.get('SUPABASE_SECRET_KEY')
    !python training/fetch_donations.py --min-score 10
except Exception as e:
    print('No shared drawings used:', e)


In [ ]:
%cd /content/tilt-sketch
import os
!python training/build_dataset.py --variants 1 --workers {os.cpu_count()}

In [ ]:
%cd /content/tilt-sketch
!python training/train.py --epochs 12 --batch 512 --arch wide --fast

In [ ]:
%cd /content/tilt-sketch
!python training/evaluate.py
!cat docs/EVALUATION.md

## Bring the model into the game
Download the zip below and send it back (or upload the three files to the repository): `models/tiltsketch/spec.json`, `models/tiltsketch/weights.bin` and `docs/EVALUATION.md`. The game loads the new model automatically when those files are present.

In [ ]:
%cd /content/tilt-sketch
!zip -r tiltsketch-model.zip models/tiltsketch docs/EVALUATION.md
from google.colab import files
files.download('tiltsketch-model.zip')